# 01 — Export Sensitivity-Guided Targeted QAT Checkpoint (Qwen3-1.7B)

Adapts `quant research/notebooks/15_mixed_precision_guided_targeted_qat.ipynb`
(Stage 1: single `oneshot()` GPTQ call with `config_groups`, W4 on the ~15% most
sensitive/"protected" layers, W3 on the rest; Stage 2: targeted QAT fine-tuning of
only the still-W3 layers via a custom `FakeQuantize` STE module) and, unlike notebook
15, actually **persists a checkpoint** via a real compressed export
(`save_pretrained(..., save_compressed=True)`), so the on-disk/VRAM footprint
reflects genuine W3/W4 savings rather than a full-precision `save_pretrained()`.

Default calibration seed: 42 (the prior project's "verified clean" flagship seed).
Output: `checkpoints/qwen3-1.7b-sgt-qat/` (HF-format, loadable directly by vLLM as a
`method="draft_model"` drafter — see notebook 03 and `docs/context.md` for the
confirmed `speculative_config` shape).

**Not yet run.** Needs a Colab Pro GPU session to execute and validate — in
particular, confirm that `save_compressed=True` correctly re-quantizes the
Stage-2-trained float weights onto the grid implied by their stored `weight_scale`/
`weight_zero_point` (rather than silently saving them uncompressed), and sanity-check
the resulting perplexity against notebook 15's Stage 1+2 numbers before trusting this
checkpoint in a benchmark.

## Setup

In [ ]:
# Clone this repo so checkpoints/, results/, and notebooks/common/ resolve correctly
# below. Skips the clone if already run once in this session (Colab keeps the runtime
# alive between cell re-runs, so re-cloning on every execution isn't necessary).
import os
REPO_NAME = 'sgt-qat-draft'
if not os.path.isdir(REPO_NAME):
    !git clone https://github.com/Resh19S/sgt-qat-draft.git
%cd {REPO_NAME}

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!pip install -q llmcompressor bitsandbytes
!pip uninstall -y scikit-learn scipy torchao -q

import sys, torch, json, math, gc
from pathlib import Path
from datetime import datetime, timezone
import torch.nn as nn
from datasets import load_dataset, Dataset

print(torch.__version__)
assert torch.cuda.is_available(), "No GPU detected."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

In [ ]:
MODEL_ID = 'Qwen/Qwen3-1.7B'
CALIB_SEED = 42          # prior project's verified-clean flagship seed
SEQ_LEN = 2048
CALIB_N = 128
PROTECT_FRAC = 0.15      # fraction of quantized params protected at W4, same as flagship run

REPO_DIR = Path('.').resolve()  # the cloned repo root, from the cell above
CHECKPOINT_DIR = REPO_DIR / 'checkpoints' / 'qwen3-1.7b-sgt-qat'
RESULTS_DIR = REPO_DIR / 'results'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Reference PPL from the prior project (quant research/notebooks/15), seed=42, for a
# sanity check after Stage 1+2 below — our numbers should land close to these.
PRIOR_PROJECT_REFERENCE = {
    'fp16_ppl': 16.67,
    'ptq_w3_seed42': 28.50,
    'full_qat_baseline_ppl_seed42': 17.53,
}

In [ ]:
import transformers.tokenization_utils_base as _tub
if hasattr(_tub, 'list_repo_templates'):
    _original_list_repo_templates = _tub.list_repo_templates
    def _safe_list_repo_templates(*args, **kwargs):
        try:
            return _original_list_repo_templates(*args, **kwargs)
        except Exception:
            return []
    _tub.list_repo_templates = _safe_list_repo_templates

from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)


def get_calibration_data(tokenizer, n=CALIB_N, seq_len=SEQ_LEN, seed=CALIB_SEED):
    ds = load_dataset('allenai/c4', 'en', split='train', streaming=True)
    ds = ds.shuffle(seed=seed, buffer_size=10_000)
    samples, collected = [], 0
    for item in ds:
        enc = tokenizer(item['text'], return_tensors='pt', truncation=True, max_length=seq_len)
        if enc['input_ids'].shape[1] == seq_len:
            samples.append(enc['input_ids'])
            collected += 1
            if collected >= n:
                break
    return torch.cat(samples, dim=0)


def compute_perplexity_wikitext2(model, tokenizer, seq_len=SEQ_LEN, stride=2048):
    ds = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='test')
    text = '\n\n'.join(ds['text'])
    enc = tokenizer(text, return_tensors='pt')
    input_ids = enc['input_ids'].to(model.device)
    seq_len_total = input_ids.size(1)
    nlls, prev_end = [], 0
    model.eval()
    for begin in range(0, seq_len_total - seq_len, stride):
        end = begin + seq_len
        target_len = end - max(begin, prev_end)
        with torch.no_grad():
            out = model(input_ids[:, begin:end], labels=input_ids[:, begin:end])
        nlls.append(out.loss * target_len)
        prev_end = end
    return float(math.exp(torch.stack(nlls).sum() / prev_end))

## 1. Per-layer sensitivity ranking

No cached ranking exists in this new repo (unlike notebook 15, which could reuse a
prior Drive-cached ranking) — always computed fresh here.

In [ ]:
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier


def build_gptq_checkpoint(model_id, tokenizer, bits, seed=CALIB_SEED):
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='cuda', trust_remote_code=True)
    calib_tensor = get_calibration_data(tokenizer, seed=seed)
    calib_dataset = Dataset.from_dict({'input_ids': calib_tensor.tolist()})
    if bits == 4:
        recipe = GPTQModifier(targets='Linear', ignore=['lm_head'], scheme='W4A16', dampening_frac=0.01)
    else:
        recipe = GPTQModifier(dampening_frac=0.01, ignore=['lm_head'],
            config_groups={'group_0': {'targets': ['Linear'], 'input_activations': None, 'output_activations': None,
                                        'weights': {'num_bits': bits, 'type': 'int', 'symmetric': True, 'strategy': 'group', 'group_size': 128}}})
    oneshot(model=model, dataset=calib_dataset, recipe=recipe, max_seq_length=SEQ_LEN, num_calibration_samples=CALIB_N)
    return model


def build_probe_ids(tokenizer, seq_len=SEQ_LEN, n_windows=4):
    ds = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='test')
    text = '\n\n'.join(ds['text'])
    enc = tokenizer(text, return_tensors='pt')
    return enc['input_ids'][:, :n_windows * seq_len]


def compute_perplexity_probe(model, probe_ids, seq_len=SEQ_LEN):
    ids = probe_ids.to(model.device)
    total_len = ids.size(1)
    nlls, prev_end = [], 0
    model.eval()
    for begin in range(0, total_len - seq_len + 1, seq_len):
        end = begin + seq_len
        target_len = end - max(begin, prev_end)
        with torch.no_grad():
            out = model(ids[:, begin:end], labels=ids[:, begin:end])
        nlls.append(out.loss * target_len)
        prev_end = end
    return float(math.exp(torch.stack(nlls).sum() / prev_end))


def get_module(model, name):
    mod = model
    for part in name.split('.'):
        mod = getattr(mod, part)
    return mod


print("Building throwaway W3/W4 checkpoints to derive the sensitivity ranking...")
model_w3_tmp = build_gptq_checkpoint(MODEL_ID, tokenizer, bits=3, seed=CALIB_SEED)
model_w4_tmp = build_gptq_checkpoint(MODEL_ID, tokenizer, bits=4, seed=CALIB_SEED)
probe_ids = build_probe_ids(tokenizer)
ppl_w3_probe = compute_perplexity_probe(model_w3_tmp, probe_ids)

layer_names_tmp = [name for name, module in model_w3_tmp.named_modules()
                   if isinstance(module, nn.Linear) and hasattr(module, 'weight_scale')]
sensitivity = []
for name in layer_names_tmp:
    dst = get_module(model_w3_tmp, name)
    src = get_module(model_w4_tmp, name)
    snap = dst.weight.data.clone()
    dst.weight.data.copy_(src.weight.data)
    hybrid_ppl = compute_perplexity_probe(model_w3_tmp, probe_ids)
    dst.weight.data.copy_(snap)
    n_params = dst.weight.numel()
    improvement = ppl_w3_probe - hybrid_ppl
    sensitivity.append({'layer': name, 'n_params': n_params,
                        'value_per_mparam': improvement / (n_params / 1e6) if n_params else 0})
sensitivity.sort(key=lambda d: d['value_per_mparam'], reverse=True)
del model_w3_tmp, model_w4_tmp
gc.collect(); torch.cuda.empty_cache()

print(f"Computed sensitivity ranking for {len(sensitivity)} layers.")
for d in sensitivity[:10]:
    print(f"  {d['layer']:<45} value/Mparam={d['value_per_mparam']:.4f}")

## 2. Stage 1 — genuine mixed-precision GPTQ (one `oneshot()` call)

In [ ]:
ranked = sorted(sensitivity, key=lambda d: d['value_per_mparam'], reverse=True)
total_params = sum(d['n_params'] for d in ranked)
target_params = PROTECT_FRAC * total_params

protected_layers, cumulative_params = [], 0
for d in ranked:
    if cumulative_params >= target_params:
        break
    protected_layers.append(d['layer'])
    cumulative_params += d['n_params']

actual_frac = cumulative_params / total_params
print(f"Protecting {len(protected_layers)} layers at W4 ({actual_frac*100:.1f}% of quantized params)")

_probe_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='cpu', trust_remote_code=True)
all_linear_names = [name for name, module in _probe_model.named_modules()
                    if isinstance(module, nn.Linear) and name != 'lm_head']
del _probe_model
unprotected_layers = [n for n in all_linear_names if n not in set(protected_layers)]
print(f"{len(protected_layers)} layers -> W4, {len(unprotected_layers)} layers -> W3 (of {len(all_linear_names)} total)")

model_mixed = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='cuda', trust_remote_code=True)
calib_tensor = get_calibration_data(tokenizer, seed=CALIB_SEED)
calib_dataset = Dataset.from_dict({'input_ids': calib_tensor.tolist()})

mixed_recipe = GPTQModifier(
    dampening_frac=0.01, ignore=['lm_head'],
    config_groups={
        'w4_protected': {
            'targets': protected_layers,
            'input_activations': None, 'output_activations': None,
            'weights': {'num_bits': 4, 'type': 'int', 'symmetric': True, 'strategy': 'group', 'group_size': 128},
        },
        'w3_rest': {
            'targets': unprotected_layers,
            'input_activations': None, 'output_activations': None,
            'weights': {'num_bits': 3, 'type': 'int', 'symmetric': True, 'strategy': 'group', 'group_size': 128},
        },
    },
)
oneshot(model=model_mixed, dataset=calib_dataset, recipe=mixed_recipe, max_seq_length=SEQ_LEN, num_calibration_samples=CALIB_N)

ppl_mixed_stage1 = compute_perplexity_wikitext2(model_mixed, tokenizer)
print(f"Stage 1 (mixed precision alone) PPL: {ppl_mixed_stage1:.2f}")
print(f"Reference (prior project, pure W3 seed=42): {PRIOR_PROJECT_REFERENCE['ptq_w3_seed42']:.2f}")

## 3. Stage 2 — targeted QAT on the still-W3 layers only

In [ ]:
import bitsandbytes as bnb

class FakeQuantize(nn.Module):
    def __init__(self, bits=3):
        super().__init__()
        self.qmin = -(2 ** (bits - 1))
        self.qmax = (2 ** (bits - 1)) - 1

    def forward(self, x, scale, zero_point):
        out_features, in_features = x.shape
        num_groups = scale.shape[-1]
        group_size = in_features // num_groups
        x_grouped = x.reshape(out_features, num_groups, group_size)
        scale_b = scale.unsqueeze(-1).to(x.dtype)
        zp_b = zero_point.unsqueeze(-1).to(x.dtype)
        x_q = ((x_grouped / scale_b) + zp_b).round().clamp(self.qmin, self.qmax)
        x_dq = (x_q - zp_b) * scale_b
        x_ste = x_grouped + (x_dq - x_grouped).detach()
        return x_ste.reshape(out_features, in_features)

def insert_fake_quant_on_subset(model, layer_names, bits=3):
    handles = []
    fq = FakeQuantize(bits=bits)
    target_set = set(layer_names)
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and hasattr(module, 'weight_scale') and name in target_set:
            def make_hook(m):
                def hook(mod, inp):
                    mod.weight.data = fq(mod.weight.data, mod.weight_scale, mod.weight_zero_point)
                return hook
            handles.append(module.register_forward_pre_hook(make_hook(module)))
    return handles

LR = 1e-5
TRAIN_STEPS = 500
BATCH_TOKENS = 1024

ds_train = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train')
train_text = '\n\n'.join(ds_train['text'])
all_ids = tokenizer(train_text, return_tensors='pt')['input_ids'][0].to('cuda')

# 1.7B QAT uses fp32 (T4/L4 lack bf16 tensor cores; raw fp16 diverges), per the prior
# project's project-wide precision protocol (quant research/notebooks/03).
QAT_PRECISION = 'fp32'
model_mixed = model_mixed.float()

fq_handles = insert_fake_quant_on_subset(model_mixed, unprotected_layers, bits=3)
n_trainable = 0
for name, module in model_mixed.named_modules():
    if isinstance(module, nn.Linear) and hasattr(module, 'weight_scale'):
        module.weight_scale.requires_grad_(False)
        is_still_w3 = name in set(unprotected_layers)
        module.weight.requires_grad_(is_still_w3)
        if is_still_w3:
            n_trainable += module.weight.numel()
print(f"Trainable parameters (still-W3 layers only): {n_trainable/1e6:.1f}M")

trainable_params = [p for p in model_mixed.parameters() if p.requires_grad]
optimizer = bnb.optim.AdamW8bit(trainable_params, lr=LR)
model_mixed.train()
if hasattr(model_mixed, 'gradient_checkpointing_enable'):
    model_mixed.gradient_checkpointing_enable()

for step in range(TRAIN_STEPS):
    start = (step * BATCH_TOKENS) % (len(all_ids) - BATCH_TOKENS - 1)
    chunk = all_ids[start:start + BATCH_TOKENS].unsqueeze(0)
    optimizer.zero_grad()
    out = model_mixed(chunk, labels=chunk)
    if not torch.isfinite(out.loss):
        raise RuntimeError(f"Loss non-finite at step {step}")
    out.loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
    optimizer.step()
    if step % 100 == 0:
        print(f"  step {step}/{TRAIN_STEPS}  loss={out.loss.item():.4f}")

for h in fq_handles:
    h.remove()
del optimizer
model_mixed.zero_grad(set_to_none=True)
gc.collect(); torch.cuda.empty_cache()

ppl_combined = compute_perplexity_wikitext2(model_mixed, tokenizer)
print(f"Stage 1+2 (mixed precision + targeted QAT) PPL: {ppl_combined:.2f}")
print(f"Reference (prior project, full-parameter QAT seed=42): {PRIOR_PROJECT_REFERENCE['full_qat_baseline_ppl_seed42']:.2f}")

## 4. Compressed export

**This is the step notebook 15 never did.** `save_compressed=True` should re-quantize
each Linear's final (Stage-2-trained) weight onto the grid implied by its stored
`weight_scale`/`weight_zero_point` and pack it (int4/int3), rather than writing raw
fp32 tensors. Verify this assumption against the actual llmcompressor/compressed-tensors
version installed — if `save_compressed=True` isn't available or behaves differently,
that's a real blocker to resolve here, not downstream.

In [ ]:
model_mixed = model_mixed.half()  # cast back before compressed export
model_mixed.save_pretrained(str(CHECKPOINT_DIR), save_compressed=True)
tokenizer.save_pretrained(str(CHECKPOINT_DIR))

def _dir_size_bytes(path):
    return sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file())

checkpoint_size_mb = _dir_size_bytes(CHECKPOINT_DIR) / (1024 ** 2)
print(f"Exported checkpoint size: {checkpoint_size_mb:.0f} MB (compare against fp16 Qwen3-1.7B ≈ 3.4 GB as a sanity check — should be substantially smaller)")

## 5. Log export metadata to results/

In [ ]:
export_record = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'experiment': 'export_sgt_qat_checkpoint',
    'model': MODEL_ID,
    'calib_seed': CALIB_SEED,
    'protect_frac_target': PROTECT_FRAC,
    'protect_frac_actual': round(actual_frac, 4),
    'n_layers_protected_w4': len(protected_layers),
    'n_layers_w3': len(unprotected_layers),
    'ppl_mixed_stage1_only': ppl_mixed_stage1,
    'ppl_combined_stage1_plus_2': ppl_combined,
    'prior_project_reference': PRIOR_PROJECT_REFERENCE,
    'n_trainable_params_stage2': n_trainable,
    'qat_steps': TRAIN_STEPS,
    'qat_lr': LR,
    'qat_precision': QAT_PRECISION,
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'checkpoint_size_mb': round(checkpoint_size_mb, 1),
    'save_compressed': True,
    'gpu': torch.cuda.get_device_name(0),
}

ts = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%S')
fname = f"export_sgt_qat_checkpoint_seed{CALIB_SEED}_{ts}.json"
(RESULTS_DIR / fname).write_text(json.dumps(export_record, indent=2))
print(f"Saved: results/{fname}")
print("\nRemember to add a docs/findings.md and docs/logs.md entry once this has actually been run.")